In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_legal_action_qca(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "LegalActionID String, SourceSystem String, PortfolioKey String, PortfolioName String, IngestionTimestamp timestamp,"
        #     "BatchId String, LegalActionKey String, LegalActionCode String, ExposureKey String, PartyKey String,"
        #     "CDI String, AgreementKey String,CaseCode string, CaseCodePrimary string, LegalActionCreatedDate timestamp, LegalActionCreatedBy String,"
        #     "LegalActionDate timestamp, LegalActionTypeCode String,LegalActionType String, LegalActionStatusCode String,"
        #     "LegalActionStatus String, LegalActionStepKey String, LegalActionStepTypeCode String,"
        #     "LegalActionStepType String, LegalActionStepStartDate timestamp,LegalActionStepEndDate timestamp,"
        #     "LegalTypeCode string,LegalType String, CompletionResultCode String, CompletionResultDate timestamp, CompletionResult String,"
        #     "LawyerPartyKey String,LawyerPartyName string, BankAccount string, Comments String, BusinessDate date"
        # )
        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.LegalAction"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qca",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== fs_activity_leaf ==================

        fs_activity_leaf_extra_filters = None

        if fs_activity_leaf_extra_filters is not None:
            fs_activity_leaf_extra_filters.append(date_filter)
            fs_activity_leaf_final_filters = " AND ".join(
                fs_activity_leaf_extra_filters
            )
        else:
            fs_activity_leaf_final_filters = date_filter

        fs_activity_leaf_selected_fields = [
            "CASE_ID",
            "ACTV_ID",
            "CAC_CODE",
            "CAC_CR_DATE",
            "PD_DESC",
            "EXTC_ID",
            "COL_ID",
            "SPET_ID",
            "businessdate",
        ]

        fs_activity_leaf = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_activity_leaf",
            fs_activity_leaf_final_filters,
            fs_activity_leaf_selected_fields,
        )

        # ================== fs_cases_snap ==================

        # fs_cases_snap_extra_filters = None

        # if fs_cases_snap_extra_filters is not None:
        #    fs_cases_snap_extra_filters.append(date_filter)
        #    fs_cases_snap_final_filters = " AND ".join(fs_cases_snap_extra_filters)
        # else:
        #    fs_cases_snap_final_filters = date_filter

        # fs_cases_snap_selected_fields = [
        #    "CASE_CODE",
        #    "CSTM_TEXT_10",
        #    "SPET_CODE",
        # ]

        # fs_cases_snap = load_landed_delta_table(
        #    spark,
        #    catalog_landed,  # type: ignore
        #    "qca_fs_cases_snap",
        #    fs_cases_snap_final_filters,
        #    fs_cases_snap_selected_fields,
        # )

        # ================== dim_cases_static ==================

        dim_cases_static_extra_filters = None

        if dim_cases_static_extra_filters is not None:
            dim_cases_static_extra_filters.append(date_filter)
            dim_cases_static_final_filters = " AND ".join(
                dim_cases_static_extra_filters
            )
        else:
            dim_cases_static_final_filters = date_filter

        dim_cases_static_selected_fields = [
            "CASE_CODE",
            "CASE_ACCOUNT_FOR_SEARCH",
            "CUST_CODE",
            "CASE_CODE_PRIMARY",
            "CASE_CONTRACT_ACCOUNT_KEY",
        ]

        dim_cases_static = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_cases_static",
            dim_cases_static_final_filters,
            dim_cases_static_selected_fields,
        )

        # ================== dim_collector ==================

        dim_collector_extra_filters = None

        if dim_collector_extra_filters is not None:
            dim_collector_extra_filters.append(date_filter)
            dim_collector_final_filters = " AND ".join(dim_collector_extra_filters)
        else:
            dim_collector_final_filters = date_filter

        dim_collector_selected_fields = [
            "col_Id",
        ]

        dim_collector = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_collector",
            dim_collector_final_filters,
            dim_collector_selected_fields,
        )

        # ================== dim_activity_type ==================

        dim_activity_type_extra_filters = None

        if dim_activity_type_extra_filters is not None:
            dim_activity_type_extra_filters.append(date_filter)
            dim_activity_type_final_filters = " AND ".join(
                dim_activity_type_extra_filters
            )
        else:
            dim_activity_type_final_filters = date_filter

        dim_activity_type_selected_fields = [
            "ACTV_ID",
            "IS_LEGAL",
            "ACTV_INTCODE",
            "ACTV_DESC",
        ]

        dim_activity_type = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_activity_type",
            dim_activity_type_final_filters,
            dim_activity_type_selected_fields,
        )
        # ================== dim_external_company ==================

        dim_external_company_extra_filters = None

        if dim_external_company_extra_filters is not None:
            dim_external_company_extra_filters.append(date_filter)
            dim_external_company_final_filters = " AND ".join(
                dim_external_company_extra_filters
            )
        else:
            dim_external_company_final_filters = date_filter

        dim_external_company_selected_fields = [
            "EXTC_ID",
            "EXTC_NAME",
        ]

        dim_external_company = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_external_company",
            dim_external_company_final_filters,
            dim_external_company_selected_fields,
        )
        # ================== dim_customer_static ==================

        dim_customer_static_extra_filters = None

        if dim_customer_static_extra_filters is not None:
            dim_customer_static_extra_filters.append(date_filter)
            dim_customer_static_final_filters = " AND ".join(
                dim_customer_static_extra_filters
            )
        else:
            dim_customer_static_final_filters = date_filter

        dim_customer_static_selected_fields = [
            "cust_cif",
            "cust_id",
        ]

        dim_customer_static = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_customer_static",
            dim_customer_static_final_filters,
            dim_customer_static_selected_fields,
        )


        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = ["PortfolioName", "portfolio_id_qca"]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )
        # ================== Joins ==================

        fs_activity_leaf_dim_cases_static_join_condition = (
            fs_activity_leaf["CASE_ID"] == dim_cases_static["CASE_CODE"]
        )

        dim_cases_static_dim_customer_static_join_condition = (
            dim_cases_static["CUST_CODE"] == dim_customer_static["cust_id"]
        )

        # dim_cases_static_fs_cases_snap_join_condition = (
        #    dim_cases_static["CASE_CODE"] == fs_cases_snap["CASE_CODE"]
        # )

        fs_activity_leaf_dim_activity_type_join_condition = (
            fs_activity_leaf["ACTV_ID"] == dim_activity_type["ACTV_ID"]
        ) & (dim_activity_type["IS_LEGAL"] == 1)

        fs_activity_leaf_dim_collector_join_condition = (
            fs_activity_leaf["COL_ID"] == dim_collector["COL_ID"]
        )

        fs_activity_leaf_dim_external_company_join_condition = (
            fs_activity_leaf["EXTC_ID"] == dim_external_company["EXTC_ID"]
        )

        fs_activity_leaf_portfolio_join_condition = (
            fs_activity_leaf["SPET_ID"] == portfolio["portfolio_id_qca"]
        )

        target_table_qca = (
            fs_activity_leaf.join(
                dim_cases_static,
                fs_activity_leaf_dim_cases_static_join_condition,
                "INNER",
            )
            .join(
                dim_customer_static,
                dim_cases_static_dim_customer_static_join_condition,
                "INNER",
            )
            .join(
                dim_activity_type,
                fs_activity_leaf_dim_activity_type_join_condition,
                "INNER",
            )
            .join(dim_collector, fs_activity_leaf_dim_collector_join_condition, "LEFT")
            .join(
                dim_external_company,
                fs_activity_leaf_dim_external_company_join_condition,
                "left",
            )
            .join(
                portfolio,
                fs_activity_leaf_portfolio_join_condition,
                "left",
            )
        )

        # HashKeys
        target_table_qca_LegalActionID_primary_key_fields = [
            fs_activity_leaf.CAC_CODE,
            lit("qca"),
        ]

        target_table_qca = target_table_qca.withColumn(
            "LegalActionID",
            sha2(
                concat_ws(";", *target_table_qca_LegalActionID_primary_key_fields),
                256,
            ),
        ).distinct()

        # ================== Selection of Final Columns ==================

        result_df_qca = target_table_qca.select(
            target_table_qca.LegalActionID.alias("LegalActionID"),
            lit("qca").cast("String").alias("SourceSystem"),
            portfolio.portfolio_id_qca.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            fs_activity_leaf.CAC_CODE.cast("String").alias("LegalActionKey"),
            fs_activity_leaf.CAC_CODE.cast("String").alias("LegalActionCode"),
            dim_cases_static.CASE_ACCOUNT_FOR_SEARCH.alias("ExposureKey"),
            dim_cases_static.CUST_CODE.cast("string").alias("PartyKey"),
            dim_customer_static.cust_cif.cast("String").alias("CDI"),
            dim_cases_static.CASE_CONTRACT_ACCOUNT_KEY.cast("String").alias(
                "AgreementKey"
            ),
            dim_cases_static.CASE_CODE.cast("String").alias("CaseCode"),
            dim_cases_static.CASE_CODE_PRIMARY.cast("String").alias("CaseCodePrimary"),
            fs_activity_leaf.CAC_CR_DATE.cast("timestamp").alias(
                "LegalActionCreatedDate"
            ),
            dim_collector.col_Id.cast("String").alias("LegalActionCreatedBy"),
            fs_activity_leaf.PD_DESC.cast("timestamp").alias("LegalActionDate"),
            dim_activity_type.ACTV_INTCODE.cast("String").alias("LegalActionTypeCode"),
            dim_activity_type.ACTV_DESC.cast("String").alias("LegalActionType"),
            lit(None).cast("String").alias("LegalActionStatusCode"),
            lit(None).cast("String").alias("LegalActionStatus"),
            lit(None).cast("String").alias("LegalActionStepKey"),
            lit(None).cast("String").alias("LegalActionStepTypeCode"),
            lit(None).cast("String").alias("LegalActionStepType"),
            lit(None).cast("timestamp").alias("LegalActionStepStartDate"),
            lit(None).cast("timestamp").alias("LegalActionStepEndDate"),
            lit(None).cast("string").alias("LegalTypeCode"),
            lit(None).cast("String").alias("LegalType"),
            lit(None).cast("String").alias("CompletionResultCode"),
            lit(None).cast("timestamp").alias("CompletionResultDate"),
            lit(None).cast("String").alias("CompletionResult"),
            fs_activity_leaf.EXTC_ID.cast("String").alias("LawyerPartyKey"),
            dim_external_company.EXTC_NAME.cast("string").alias("LawyerPartyName"),
            lit(None).cast("String").alias("BankAccount"),
            lit(None).cast("String").alias("Comments"),
            fs_activity_leaf.businessdate.cast("date").alias("BusinessDate"),
        )

        # ================== Append ==================

        result_df_qca.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_legal_action_qca(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)